# ENSF617 Main Training Notebook

This notebook uses the shared helpers in `main.py` to prepare data, train the model, run test evaluation, and optionally save predictions.

**How to use it:** edit the configuration cell, run the config-building cell, then run the workflow cell.

**Disclaimer:** this notebook is intended for research and development use inside the repository. The default configuration is a baseline starting point, not a claim of optimal performance or clinical validity.

In [ ]:
# Notebook bootstrap:
# - assume the notebook is launched from the repository root
# - make the repo root importable so `defaults.py` and `main.py` can be imported
# - reuse the same shared helpers as the CLI entrypoint
from pathlib import Path
import sys

ROOT_DIR = Path.cwd()
assert (ROOT_DIR / "main.py").exists(), "Run this notebook from the repository root."
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from defaults import (
    DEFAULT_AZT1D_URL,
    build_default_config,
    build_default_observability_config,
    build_default_snapshot_config,
    build_default_train_config,
)
from config import (
    Config,
    collect_runtime_diagnostics,
    detect_runtime_environment,
    format_runtime_diagnostics,
    resolve_device_profile,
)
from main import (
    run_training_workflow,
)


In [ ]:
# Edit these values for your run.
#
# These are intentionally plain notebook variables rather than a single
# nested dict so it stays easy to tweak one thing at a time while
# experimenting.
output_dir = ROOT_DIR / "artifacts" / "notebook_run"
dataset_url = DEFAULT_AZT1D_URL
processed_dir = ROOT_DIR / "data" / "processed"
processed_file_name = "azt1d_processed.csv"

encoder_length = 168
prediction_length = 12
batch_size = 64
num_workers = 0
pin_memory = None
persistent_workers = None

max_epochs = 20
device_profile = "auto"
accelerator = "auto"
devices = "auto"
precision = 32
enable_progress_bar = None
enable_rich_progress_bar = None
enable_device_stats = None
fail_on_preflight_errors = True

learning_rate = 1e-3
weight_decay = 0.0
optimizer_name = "adam"
seed = 42

skip_test = False
skip_predict = False
save_predictions = True


In [ ]:
# Convert the editable notebook variables into the typed config objects
# used by the rest of the codebase.
#
# Keeping this step explicit makes it easier to inspect what exact config
# will be handed to the training workflow before launching a longer run.
config = build_default_config(
    dataset_url=dataset_url,
    raw_dir=ROOT_DIR / "data" / "raw",
    cache_dir=ROOT_DIR / "data" / "cache",
    extracted_dir=ROOT_DIR / "data" / "extracted",
    processed_dir=processed_dir,
    processed_file_name=processed_file_name,
    encoder_length=encoder_length,
    prediction_length=prediction_length,
    batch_size=batch_size,
    num_workers=num_workers,
    pin_memory=False if pin_memory is None else pin_memory,
    persistent_workers=(
        False if persistent_workers is None else persistent_workers
    ),
)

train_config = build_default_train_config(
    max_epochs=max_epochs,
    accelerator=accelerator,
    devices=devices,
    precision=precision,
    enable_progress_bar=(
        True if enable_progress_bar is None else enable_progress_bar
    ),
    default_root_dir=output_dir,
)

observability_config = build_default_observability_config(
    output_dir=output_dir,
    enable_device_stats=(
        True if enable_device_stats is None else enable_device_stats
    ),
    enable_rich_progress_bar=(
        True if enable_rich_progress_bar is None else enable_rich_progress_bar
    ),
)

snapshot_config = build_default_snapshot_config(output_dir=output_dir)
runtime_environment = detect_runtime_environment()
profile_resolution = resolve_device_profile(
    requested_profile=device_profile,
    environment=runtime_environment,
    train_config=train_config,
    data_config=config.data,
    observability_config=observability_config,
)
config = Config(
    data=profile_resolution.data_config,
    tft=config.tft,
    tcn=config.tcn,
)
train_config = profile_resolution.train_config
observability_config = profile_resolution.observability_config
preflight_diagnostics = collect_runtime_diagnostics(
    requested_profile=profile_resolution.requested_profile,
    resolved_profile=profile_resolution.resolved_profile,
    environment=runtime_environment,
    train_config=train_config,
    data_config=config.data,
    observability_config=observability_config,
)

# Display the resolved runtime state so the notebook user can sanity-check it.
{
    "requested_profile": profile_resolution.requested_profile,
    "resolved_profile": profile_resolution.resolved_profile,
    "applied_defaults": profile_resolution.applied_defaults,
    "diagnostics": format_runtime_diagnostics(preflight_diagnostics),
}


In [ ]:
# Run the shared end-to-end workflow.
#
# This cell is intentionally thin: it calls the same Python function used
# by `main.py`, which helps keep notebook and script behavior aligned.
artifacts = run_training_workflow(
    config,
    train_config=train_config,
    snapshot_config=snapshot_config,
    observability_config=observability_config,
    requested_device_profile=profile_resolution.requested_profile,
    resolved_device_profile=profile_resolution.resolved_profile,
    applied_profile_defaults=profile_resolution.applied_defaults,
    runtime_environment=runtime_environment,
    preflight_diagnostics=preflight_diagnostics,
    fail_on_preflight_errors=fail_on_preflight_errors,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    optimizer_name=optimizer_name,
    output_dir=output_dir,
    seed=seed,
    skip_test=skip_test,
    skip_predict=skip_predict,
    save_predictions=save_predictions,
)


# Show the compact JSON-friendly summary written for this run.
artifacts.summary


In [ ]:
# Held-out test metrics, if a test split was available and `skip_test`
# was left as False.
artifacts.test_metrics

In [ ]:
# Inspect one raw prediction batch.
#
# The saved tensors are left in their direct model-output form so later
# notebook cells can decide how to aggregate, visualize, or calibrate
# them.
if artifacts.test_predictions:
    first_batch = artifacts.test_predictions[0]
    print(first_batch.shape)
    first_batch
else:
    print("No test predictions were produced for this run.")
